In [ ]:
import json
import time
import os
from pathlib import Path
import kaggle_benchmarks as kbench

In [ ]:
@kbench.task(name=f"Extract Keywords", description=f"Trích xuất từ khóa (xử lý từng event)")
def extract_keywords_task(llm) -> None:

    # Source transcripts đã được group event và sửa lỗi (nếu có) từ bước trước
    base_input_dir = f"./fixed_video_transcripts"
    base_output_dir = f"./keyword_video_transcripts"

    target_path = f"{base_input_dir}/transcripts"

    all_json_paths = []
    if os.path.exists(target_path):
        for file in os.listdir(target_path):
            if file.endswith(".json"):
                all_json_paths.append(os.path.join(target_path, file))
    else:
        print(f"❌ Không tìm thấy thư mục: {target_path}. Hãy kiểm tra lại LEVEL_NUMBER!")
        return

    all_json_paths.sort()
    total_files = len(all_json_paths)
    
    print(f"Starting extract_keywords_task. Tìm thấy tổng cộng {total_files} file JSON.")
    print("----------------------------------------------------------------------")

    for idx, input_path in enumerate(all_json_paths, 1):
        relative_path = os.path.relpath(input_path, base_input_dir)
        output_path = os.path.join(base_output_dir, relative_path)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        print(f"🎬 [{idx}/{total_files}] Đang xử lý: {relative_path}")

        try:
            with open(input_path, "r", encoding="utf-8") as f:
                transcript = json.load(f)
        except Exception as e:
            print(f"    ❌ Lỗi đọc file: {e}. Bỏ qua file này.")
            continue

        all_events = transcript.get('events', [])
        total_events = len(all_events)

        if total_events == 0:
            print(f"    ⏩ File trống (0 events). Ghi nhận và copy nguyên bản sang output.")
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(transcript, f, ensure_ascii=False, indent=2)
            continue

        print(f"    📦 Có {total_events} events. Bắt đầu xử lý từng event một...")

        for e_idx, ev in enumerate(all_events, 1):
            event_text = ev["text"].strip()
            prompt = ""
            # prompt được thay đổi dựa trên topic của video, check benchmark/prompts để xem chi tiết
            # cần đảm bảo phải cung cấp biến 'event_text' trong prompt để AI biết nội dung cần trích xuất từ khóa.

            max_retries = 5
            retry_count = 0
            success = False
            keywords = []

            while retry_count < max_retries and not success:
                try:
                    response_text = llm.prompt(prompt)
                    keywords = [kw.strip() for kw in response_text.split(';') if kw.strip()]

                    if (len(keywords) == 0) or (len(keywords) > 10):
                        print(f"Lưu ý… AI trả về {len(keywords)} từ khóa cho event {ev.get('event_id', e_idx)}.")

                    success = True
                    time.sleep(10.0)  # Nghỉ nhẹ tránh spam API

                except Exception as e:
                    retry_count += 1
                    print(f"    ⚠️ Lỗi event {ev.get('event_id', e_idx)} (Lần thử {retry_count}/{max_retries}): {e}")
                    if retry_count < max_retries:
                        wait_time = 20.0 * (2 ** (retry_count - 1))  # 20s -> 40s -> 80s -> 160s
                        print(f"    💤 Server quá tải hoặc phản hồi lỗi. Ngủ đông {wait_time:.0f} giây...")
                        time.sleep(wait_time)
                    else:
                        print("    ❌ Đã thử hết cách. HỦY KẾT QUẢ EVENT NÀY, để keywords rỗng!")
                        keywords = []

            ev["keywords"] = keywords

            if e_idx % 20 == 0 or e_idx == total_events:
                print(f"    ...đã xử lý {e_idx}/{total_events} events")

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(transcript, f, ensure_ascii=False, indent=2)

        print(f"    ✅ Đã lưu file có keywords.")

    print(f"Extract keywords task completed. Tất cả {total_files} file đã được xử lý và lưu vào {base_output_dir}.")
    kbench.assertions.assert_true(True, expectation=f"Done")

# Run the task with a specific LLM
extract_keywords_task.run(llm=kbench.llms["google/gemini-3.5-flash"])